# Asistente experto en analitica de futbol profesional con Gemini, RAG y LangGraph

Proyecto final de IA Generativa: construccion de un agente experto capaz de responder preguntas sobre analitica de futbol profesional usando una base de conocimiento vectorial propia, Gemini como LLM, Gemini Embeddings, ChromaDB, LangGraph y memoria conversacional.

**Dominio elegido:** scouting y analitica avanzada de jugadores de futbol profesional en las cinco grandes ligas europeas durante la temporada 2024-2025.

## Requisitos del enunciado

Este notebook se construye para cumplir los puntos obligatorios del proyecto:

- Base de conocimiento vectorial en ChromaDB usando Gemini Embeddings.
- Minimo de 3 documentos o equivalente a unas 20 paginas de texto, usando CSV y documento propio.
- Pipeline RAG: ingesta, chunking/preprocesamiento, embeddings, almacenamiento, recuperacion y generacion.
- Agente con LangGraph que combine RAG, Gemini y memoria conversacional.
- System prompt personalizado y justificado.
- Celda de interaccion por texto dentro del notebook.
- Minimo de 5 preguntas de ejemplo con respuestas documentadas.

## 1. Instalacion de dependencias

Ejecuta esta celda solo si tu entorno no tiene instaladas las librerias. En local se recomienda usar `requirements.txt`; en Google Colab puedes descomentar la instalacion.

In [35]:
# Esta celda sirve para instalar las librerias si el entorno no las tiene.
# En local normalmente se instalan desde requirements.txt.
# En Google Colab conviene descomentar la linea siguiente y ejecutarla una vez.
!pip install -q langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters langgraph chromadb python-dotenv pypdf pandas

## 2. Imports y configuracion del entorno

Las claves y configuraciones sensibles se cargan desde `.env`. No se debe hardcodear la API key en el notebook.

In [36]:
# Path permite trabajar con rutas de archivos de forma robusta en Windows, Linux o Colab.
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Cargamos el archivo .env para que Python pueda leer GOOGLE_API_KEY y otros parametros.
# override=True fuerza a usar el valor actual del .env aunque el kernel tuviera uno antiguo cargado.
load_dotenv(override=True)

# Leemos la clave de Gemini desde el .env.
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash").strip().strip('"')
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001").strip().strip('"')
CHROMA_DB_DIR = os.getenv("CHROMA_DB_DIR", "chroma_db").strip().strip('"')

# Si no hay API key, paramos el notebook con un error claro.
if not GOOGLE_API_KEY:
    raise ValueError("No se encontro GOOGLE_API_KEY. Revisa el archivo .env")

# Mostramos informacion de configuracion sin imprimir la API key completa.
print("API key cargada:", bool(GOOGLE_API_KEY))
print("Modelo Gemini:", GEMINI_MODEL)
print("Modelo embeddings:", GEMINI_EMBEDDING_MODEL)
print("Directorio Chroma:", CHROMA_DB_DIR)

API key cargada: True
Modelo Gemini: gemini-2.5-flash
Modelo embeddings: gemini-embedding-001
Directorio Chroma: chroma_db


## 3. Rutas del proyecto

Centralizamos las rutas para que el notebook sea mas mantenible y pueda ejecutarse desde la carpeta `notebooks` o desde la raiz del proyecto.

In [37]:
# Path.cwd() devuelve la carpeta desde la que se esta ejecutando el notebook.
NOTEBOOK_DIR = Path.cwd()

# Si ejecutamos el notebook desde la carpeta notebooks, la raiz del proyecto es el directorio padre.
# Si lo ejecutamos desde la raiz del proyecto, PROJECT_ROOT sera la carpeta actual.
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Carpeta donde estan los datos del proyecto.
DATA_DIR = PROJECT_ROOT / "data"

# Ruta del CSV con estadisticas de jugadores.
CSV_PATH = DATA_DIR / "players_data-2024_2025.csv"

# Ruta del glosario creado para explicar metricas de futbol.
GLOSSARY_PATH = DATA_DIR / "glosario_metricas_futbol.md"

# Ruta donde ChromaDB guardara los vectores de forma persistente.
CHROMA_PATH = PROJECT_ROOT / CHROMA_DB_DIR

# Lista de archivos obligatorios para que el notebook funcione.
required_paths = [CSV_PATH, GLOSSARY_PATH]

# Comprobamos que cada archivo existe antes de continuar.
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo requerido: {path}")

# Imprimimos las rutas para verificar que el notebook esta leyendo los archivos correctos.
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV:", CSV_PATH)
print("Glosario:", GLOSSARY_PATH)
print("ChromaDB:", CHROMA_PATH)

PROJECT_ROOT: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve
CSV: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\data\players_data-2024_2025.csv
Glosario: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\data\glosario_metricas_futbol.md
ChromaDB: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\chroma_db


## 4. Inicializacion de Gemini

Creamos dos componentes: el modelo conversacional para generar respuestas y el modelo de embeddings para indexar y recuperar informacion desde ChromaDB.

In [38]:
# Creamos el modelo de lenguaje Gemini para generar respuestas.
llm = ChatGoogleGenerativeAI(
    # Modelo definido en .env, por ejemplo gemini-2.5-flash.
    model=GEMINI_MODEL,
    # API key cargada desde .env.
    google_api_key=GOOGLE_API_KEY,
    # Temperatura baja para respuestas mas consistentes y menos creativas.
    temperature=0.2,
)

# Creamos el modelo de embeddings que convertira texto en vectores numericos.
embeddings = GoogleGenerativeAIEmbeddings(
    # Modelo de embeddings definido en .env.
    model=GEMINI_EMBEDDING_MODEL,
    # API key cargada desde .env.
    google_api_key=GOOGLE_API_KEY,
)

# Confirmamos que ambos componentes se han creado.
print("LLM inicializado:", GEMINI_MODEL)
print("Embeddings inicializados:", GEMINI_EMBEDDING_MODEL)

LLM inicializado: gemini-2.5-flash
Embeddings inicializados: gemini-embedding-001


## 5. Prueba minima del LLM

Antes de construir el RAG, verificamos que Gemini responde correctamente. Esta prueba no usa todavia la base de conocimiento; solo valida conexion y configuracion.

In [39]:
# Enviamos una pregunta simple al LLM para comprobar que Gemini responde.
respuesta = llm.invoke("Explica en una frase que significa xG en futbol.")

# Imprimimos solo el contenido textual de la respuesta.
print(respuesta.content)

xG (Goles Esperados) es una métrica que cuantifica la probabilidad de que un disparo termine en gol, indicando la calidad de la ocasión de marcar.


## 6. Carga inicial del dataset y del glosario

El CSV aporta datos estadisticos de jugadores y el glosario aporta conocimiento textual para interpretar las metricas. Ambos se usaran como base de conocimiento del RAG.

In [40]:
# Cargamos el CSV de jugadores en un DataFrame de pandas.
df = pd.read_csv(CSV_PATH)

# Leemos el glosario completo como texto.
glosario_texto = GLOSSARY_PATH.read_text(encoding="utf-8")

# Mostramos dimensiones del dataset: filas y columnas.
print("Dataset:", df.shape)

# Mostramos el numero total de columnas.
print("Columnas:", len(df.columns))

# Mostramos el tamano del glosario para comprobar que tiene contenido suficiente.
print("Longitud del glosario:", len(glosario_texto), "caracteres")

# Visualizamos algunas columnas importantes para comprobar que el CSV se cargo bien.
df[["Player", "Squad", "Comp", "Pos", "Age", "Min", "xG", "xAG", "PrgC", "PrgP"]].head()

Dataset: (2854, 267)
Columnas: 267
Longitud del glosario: 18177 caracteres


,Player,Squad,Comp,Pos,Age,Min,xG,xAG,PrgC,PrgP
0,Max Aarons,Bournemouth,eng Premier League,DF,24.0,86,0.0,0.0,1,8
1,Max Aarons,Valencia,es La Liga,"DF,MF",24.0,120,0.0,0.0,0,6
2,Rodrigo Abajas,Valencia,es La Liga,DF,21.0,65,0.1,0.0,3,2
3,James Abankwah,Udinese,it Serie A,"DF,MF",20.0,88,0.1,0.0,3,4
4,Keyliane Abdallah,Marseille,fr Ligue 1,FW,18.0,3,0.0,0.0,1,0


## 7. Revision rapida de cobertura de datos

Comprobamos que el dataset tiene suficiente volumen y variedad para justificar el proyecto: varias ligas, posiciones y miles de registros.

In [41]:
# Mostramos cuantas filas tiene el dataset.
print("Registros:", len(df))

# Contamos cuantos jugadores/registros hay por liga.
print("Ligas:")
display(df["Comp"].value_counts())

# Contamos las posiciones mas frecuentes del dataset.
print("Posiciones principales:")
display(df["Pos"].value_counts().head(15))

Registros: 2854
Ligas:


Comp
it Serie A            634
es La Liga            601
eng Premier League    574
fr Ligue 1            553
de Bundesliga         492
Name: count, dtype: int64

Posiciones principales:


Pos
DF       859
MF       589
FW       371
FW,MF    325
MF,FW    230
GK       212
DF,MF    110
MF,DF     81
DF,FW     53
FW,DF     24
Name: count, dtype: int64

## 8. Arquitectura MVP del asistente

Para mantener el proyecto simple y defendible, usaremos una arquitectura hibrida:

- **ChromaDB + Gemini Embeddings**: para recuperar conocimiento experto sobre metricas, criterios de scouting y algunas fichas de ejemplo.
- **pandas + CSV completo**: para consultas numericas exactas sobre todos los jugadores del dataset.
- **Gemini LLM**: para redactar una respuesta clara usando el contexto recuperado y, cuando aplique, los resultados calculados desde el CSV.

Esta decision evita depender de similitud semantica para filtros exactos como `xG > 15`, `Min >= 1000` o `Top 10 por xAG`, que se resuelven mejor con pandas.

## 9. System prompt personalizado

El system prompt define el rol, tono, limites y reglas del asistente. Es una parte obligatoria del enunciado y se documentara tambien en el README.

In [72]:
# Definimos el system prompt principal del asistente.
# Este texto se enviara a Gemini para indicarle como debe comportarse.
SYSTEM_PROMPT = """
Eres un asistente experto en analitica de futbol profesional, especializado en scouting, rendimiento de jugadores y metricas avanzadas.

Tu objetivo es ayudar a analizar jugadores de futbol usando dos fuentes de informacion:
1. Contexto recuperado desde ChromaDB, especialmente el glosario de metricas y fichas de scouting indexadas.
2. Resultados calculados desde el CSV con pandas cuando la pregunta requiera rankings, filtros numericos o comparaciones exactas.

Reglas de comportamiento:
- Responde en español, con tono claro, profesional y didactico.
- No inventes datos de jugadores ni valores numericos.
- Si una respuesta requiere datos exactos y no se han proporcionado resultados del CSV, indica que necesitas consultar el dataset.
- Si el contexto recuperado no contiene informacion suficiente, dilo explicitamente.
- Explica las metricas cuando sean importantes para interpretar la respuesta.
- Cuando compares jugadores, ten en cuenta posicion, minutos jugados, liga y rol.
- Prioriza respuestas breves, utiles y basadas en evidencia.

Limitaciones:
- No predigas resultados futuros como si fueran certezas.
- No des recomendaciones de apuestas.
- No afirmes que un jugador es mejor que otro sin explicar con que metricas se justifica.
""".strip()

print(SYSTEM_PROMPT)

Eres un asistente experto en analitica de futbol profesional, especializado en scouting, rendimiento de jugadores y metricas avanzadas.

Tu objetivo es ayudar a analizar jugadores de futbol usando dos fuentes de informacion:
1. Contexto recuperado desde ChromaDB, especialmente el glosario de metricas y fichas de scouting indexadas.
2. Resultados calculados desde el CSV con pandas cuando la pregunta requiera rankings, filtros numericos o comparaciones exactas.

Reglas de comportamiento:
- Responde en español, con tono claro, profesional y didactico.
- No inventes datos de jugadores ni valores numericos.
- Si una respuesta requiere datos exactos y no se han proporcionado resultados del CSV, indica que necesitas consultar el dataset.
- Si el contexto recuperado no contiene informacion suficiente, dilo explicitamente.
- Explica las metricas cuando sean importantes para interpretar la respuesta.
- Cuando compares jugadores, ten en cuenta posicion, minutos jugados, liga y rol.
- Prioriza res

### 9.1 Justificacion del system prompt

Este prompt se ha diseñado para cumplir el enunciado porque:

- Define un rol experto concreto: analista profesional de futbol.
- Obliga a usar evidencia procedente de ChromaDB y del CSV.
- Reduce alucinaciones al prohibir inventar datos numericos.
- Explica como actuar si falta informacion.
- Mantiene un tono didactico, adecuado para un usuario que quiere entender metricas y decisiones de scouting.

## 10. Preparacion de documentos para RAG

ChromaDB indexa documentos textuales, no tablas directamente. Por eso convertimos cada jugador relevante del CSV en una ficha textual de scouting con metadatos, y usamos el glosario como documento conceptual para explicar metricas.

In [42]:
!pip install -q langchain-text-splitters

In [43]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

### 10.1 Seleccion de columnas relevantes

El dataset tiene muchas columnas repetidas por bloques estadisticos. Para un MVP limpio, usamos las columnas mas interpretables para scouting. Esto reduce ruido y mejora la calidad de recuperacion.

In [44]:
# Lista de columnas que queremos usar para crear fichas de scouting.
# No usamos todas las columnas porque el CSV tiene muchas columnas duplicadas por bloques estadisticos.
SCOUTING_COLUMNS = [
    "Player", "Nation", "Pos", "Squad", "Comp", "Age", "Born", "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "xG", "npxG", "xAG", "xA", "npxG+xAG", "xG+xAG",
    "Sh", "SoT", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT", "Dist", "npxG/Sh", "G-xG", "np:G-xG",
    "Cmp", "Att", "Cmp%", "TotDist", "PrgDist", "KP", "1/3", "PPA", "CrsPA", "PrgP",
    "SCA", "SCA90", "GCA", "GCA90",
    "Tkl", "TklW", "Tkl%", "Blocks", "Int", "Tkl+Int", "Clr", "Err",
    "Touches", "Att Pen", "Carries", "PrgC", "CPA", "Succ", "Succ%", "Mis", "Dis", "Rec", "PrgR",
    "CrdY", "CrdR", "Fls", "Fld", "Off", "Recov", "Won", "Won%",
    "GA", "GA90", "SoTA", "Saves", "Save%", "CS", "CS%", "PSxG", "PSxG/SoT", "PSxG+/-", "#OPA", "#OPA/90", "AvgDist"
]

# Nos quedamos solo con las columnas que realmente existen en el CSV.
available_columns = [col for col in SCOUTING_COLUMNS if col in df.columns]

# Guardamos las columnas que no se encontraron para poder revisarlas.
missing_columns = [col for col in SCOUTING_COLUMNS if col not in df.columns]

# Mostramos cuantas columnas vamos a usar.
print("Columnas seleccionadas:", len(available_columns))

# Mostramos si alguna columna esperada no aparece en el dataset.
print("Columnas no encontradas:", missing_columns)

Columnas seleccionadas: 88
Columnas no encontradas: []


### 10.2 Limpieza ligera del CSV

Filtramos jugadores con muestra minima para evitar que el RAG recomiende perfiles basados en 2 o 3 minutos. Para el MVP usamos `Min >= 450`, equivalente a 5 partidos completos.

In [45]:
# Creamos una copia del DataFrame para no modificar df original.
df_players = df.copy()

# Convertimos columnas numericas importantes a tipo numerico.
# errors="coerce" convierte valores raros o vacios en NaN.
for col in ["Min", "90s", "Age"]:
    df_players[col] = pd.to_numeric(df_players[col], errors="coerce")

# Definimos un minimo de minutos para evitar conclusiones con muestras demasiado pequenas.
MIN_MINUTES = 450

# Para no agotar la cuota gratuita de Gemini, empezamos con un MVP reducido.
# Luego puedes subir este numero cuando el pipeline ya funcione.
MAX_PLAYER_DOCUMENTS = 25

# Filtramos jugadores con minutos suficientes.
df_players = df_players[df_players["Min"].fillna(0) >= MIN_MINUTES].copy()

# Ordenamos por minutos para priorizar jugadores con muestras mas fiables.
df_players = df_players.sort_values("Min", ascending=False)

# Nos quedamos con los primeros registros para crear una base vectorial manejable.
df_players = df_players.head(MAX_PLAYER_DOCUMENTS).reset_index(drop=True)

# Mostramos cuantos registros quedan despues del filtro y la limitacion de MVP.
print("Jugadores/registros tras filtro de minutos:", len(df_players))
print("Minutos minimos:", MIN_MINUTES)
print("Limite de documentos de jugadores:", MAX_PLAYER_DOCUMENTS)

# Visualizamos una muestra de jugadores con metricas clave.
display(df_players[["Player", "Squad", "Comp", "Pos", "Min", "xG", "xAG", "PrgC", "PrgP"]].head())


Jugadores/registros tras filtro de minutos: 25
Minutos minimos: 450
Limite de documentos de jugadores: 25


,Player,Squad,Comp,Pos,Min,xG,xAG,PrgC,PrgP
0,Mile Svilar,Roma,it Serie A,GK,3420,0.0,0.0,0,2
1,David Soria,Getafe,es La Liga,GK,3420,0.0,0.0,0,1
2,Matz Sels,Nott'ham Forest,eng Premier League,GK,3420,0.0,0.4,1,2
3,David Raya,Arsenal,eng Premier League,GK,3420,0.0,0.0,0,14
4,Obite N'Dicka,Roma,it Serie A,DF,3420,1.0,0.7,29,107


### 10.3 Funcion para convertir una fila en documento de scouting

Cada documento resume identidad, contexto y metricas clave del jugador. Los metadatos permiten filtrar o citar la fuente despues.

In [ ]:
# Funcion auxiliar para limpiar valores antes de meterlos en documentos.
def clean_value(value):
    # Si el valor es NaN, lo tratamos como ausente.
    if pd.isna(value):
        return None
    # Convertimos a texto y quitamos espacios al principio y al final.
    value = str(value).strip()
    # Devolvemos el texto si no esta vacio; si esta vacio, devolvemos None.
    return value if value else None


# Esta funcion convierte una fila del CSV en un documento textual de scouting.
def row_to_scouting_document(row):
    # Extraemos datos basicos del jugador con valores por defecto si faltan.
    player = clean_value(row.get("Player")) or "Jugador desconocido"
    squad = clean_value(row.get("Squad")) or "Equipo desconocido"
    comp = clean_value(row.get("Comp")) or "Competicion desconocida"
    pos = clean_value(row.get("Pos")) or "Posicion desconocida"

    # Aqui iremos guardando lineas de texto con las metricas disponibles.
    metric_lines = []

    # Recorremos las columnas seleccionadas para crear una ficha legible.
    for col in available_columns:
        # Limpiamos el valor de la columna actual.
        value = clean_value(row.get(col))
        # Solo incluimos metricas con valor real.
        if value is not None:
            metric_lines.append(f"- {col}: {value}")

    # Construimos el contenido textual que se guardara en ChromaDB.
    content = f"""
Ficha de scouting de {player}

Contexto:
- Jugador: {player}
- Equipo: {squad}
- Liga: {comp}
- Posicion: {pos}

Metricas disponibles:
{chr(10).join(metric_lines)}

Interpretacion recomendada:
Este documento debe usarse para responder preguntas sobre rendimiento, comparacion de jugadores y scouting. Las metricas deben interpretarse segun posicion, minutos jugados, liga y rol tactico.
""".strip()

    # Los metadatos ayudan a identificar la fuente y filtrar resultados después.
    metadata = {
        "source": "players_data-2024_2025.csv",
        "doc_type": "player_scouting_profile",
        "player": player,
        "squad": squad,
        "competition": comp,
        "position": pos,
    }

    # Devolvemos un objeto Document con texto y metadatos.
    return Document(page_content=content, metadata=metadata)

### 10.4 Creacion de documentos desde el CSV

Generamos un documento por registro de jugador filtrado. Esto permite recuperar fichas concretas cuando el usuario pregunta por jugadores, posiciones, equipos o metricas.

In [47]:
# Convertimos cada fila filtrada del CSV en un documento de LangChain.
player_documents = [
    row_to_scouting_document(row)
    for _, row in df_players.iterrows()
]

# Mostramos cuantos documentos de jugadores se han creado.
print("Documentos de jugadores:", len(player_documents))

# Imprimimos una parte del primer documento para comprobar su formato.
print(player_documents[0].page_content[:1000])

# Imprimimos los metadatos del primer documento.
print("Metadatos:", player_documents[0].metadata)

Documentos de jugadores: 25
Ficha de scouting de Mile Svilar

Contexto:
- Jugador: Mile Svilar
- Equipo: Roma
- Liga: it Serie A
- Posicion: GK

Metricas disponibles:
- Player: Mile Svilar
- Nation: rs SRB
- Pos: GK
- Squad: Roma
- Comp: it Serie A
- Age: 24.0
- Born: 1999.0
- MP: 38
- Starts: 38
- Min: 3420
- 90s: 38.0
- Gls: 0
- Ast: 0
- G+A: 0
- G-PK: 0
- PK: 0
- PKatt: 0
- xG: 0.0
- npxG: 0.0
- xAG: 0.0
- xA: 0.0
- npxG+xAG: 0.0
- xG+xAG: 0.0
- Sh: 1
- SoT: 0
- SoT%: 0.0
- Sh/90: 0.03
- SoT/90: 0.0
- G/Sh: 0.0
- Dist: 10.5
- npxG/Sh: 0.05
- G-xG: 0.0
- np:G-xG: 0.0
- Cmp: 1151
- Att: 1459
- Cmp%: 78.9
- TotDist: 29765
- PrgDist: 16946
- KP: 0
- 1/3: 15
- PPA: 0
- CrsPA: 0
- PrgP: 2
- SCA: 6
- SCA90: 0.16
- GCA: 0
- GCA90: 0.0
- Tkl: 0
- TklW: 0
- Blocks: 0
- Int: 0
- Tkl+Int: 0
- Clr: 12
- Err: 2
- Touches: 1531
- Att Pen: 1
- Carries: 944
- PrgC: 0
- CPA: 0
- Succ: 0
- Mis: 0
- Dis: 1
- Rec: 874
- PrgR: 0
- CrdY: 1
- CrdR: 0
- Fls: 0
- Fld: 0
- Off: 1
- Recov: 53
- Won: 6
- Won%: 

## 11. Chunking del glosario

El glosario contiene conocimiento conceptual. Lo dividimos en chunks para que el retriever pueda recuperar definiciones concretas sin meter todo el documento en cada prompt.

In [64]:
# Creamos un documento unico a partir del texto completo del glosario.
glossary_document = Document(
    # Texto que se va a dividir e indexar.
    page_content=glosario_texto,
    # Metadatos para saber de donde viene este documento.
    metadata={
        "source": "glosario_metricas_futbol.md",
        "doc_type": "metric_glossary",
    },
)

# Configuramos el divisor de texto.
text_splitter = RecursiveCharacterTextSplitter(
    # Tamano aproximado de cada chunk en caracteres.
    chunk_size=1000,
    # Solapamiento entre chunks para no perder contexto entre cortes.
    chunk_overlap=150,
    # Separadores preferidos: primero titulos Markdown, luego parrafos, frases y palabras.
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)

# Dividimos el glosario en fragmentos mas pequenos.
glossary_chunks = text_splitter.split_documents([glossary_document])

# Mostramos cuantos chunks se han creado.
print("Chunks del glosario:", len(glossary_chunks))

# Mostramos el inicio del primer chunk para revisar el resultado.
print(glossary_chunks[0].page_content[:1000])

# Mostramos los metadatos del primer chunk.
print("Metadatos:", glossary_chunks[0].metadata)

Chunks del glosario: 27
# Glosario de metricas para analitica de futbol profesional

Este documento define las principales metricas usadas por el asistente experto en analitica de futbol profesional. Su objetivo es ayudar al sistema RAG a interpretar correctamente las columnas del dataset `players_data-2024_2025.csv` y a responder preguntas de scouting, rendimiento y comparacion de jugadores.
Metadatos: {'source': 'glosario_metricas_futbol.md', 'doc_type': 'metric_glossary'}


In [65]:
for i, chunk in enumerate(glossary_chunks[:5], start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(chunk.page_content[:500])
    print(chunk.metadata)


Chunk 1
# Glosario de metricas para analitica de futbol profesional

Este documento define las principales metricas usadas por el asistente experto en analitica de futbol profesional. Su objetivo es ayudar al sistema RAG a interpretar correctamente las columnas del dataset `players_data-2024_2025.csv` y a responder preguntas de scouting, rendimiento y comparacion de jugadores.
{'source': 'glosario_metricas_futbol.md', 'doc_type': 'metric_glossary'}
Chunk 2
## Identificacion del jugador

### Player

Nombre del jugador.

Uso analitico: permite buscar, comparar y generar informes individuales.

### Nation

Nacionalidad del jugador.

Uso analitico: puede ser relevante para restricciones de plantilla, scouting internacional o analisis por pais.

### Pos

Posicion principal o combinada del jugador.

Ejemplos habituales:

- `GK`: portero.
- `DF`: defensa.
- `MF`: centrocampista.
- `FW`: delantero.
- `DF,MF`: jugador con uso defensivo y de mediocampo.
- `
{'source': 'glosario_metricas_futbol.m

## 12. Creacion de la base vectorial con ChromaDB

Unimos documentos de jugadores y chunks del glosario, calculamos embeddings con Gemini y persistimos la coleccion en ChromaDB. Esta es la base de conocimiento del asistente.

In [ ]:
# Unimos los documentos estadisticos de jugadores con los chunks conceptuales del glosario
rag_documents = player_documents + glossary_chunks

# Mostramos el total de documentos que se indexaran en ChromaDB
print("Total de documentos para indexar:", len(rag_documents))

# Mostramos cuantos documentos vienen del CSV
print("- Jugadores:", len(player_documents))

# Mostramos cuantos documentos vienen del glosario
print("- Chunks glosario:", len(glossary_chunks))

# Nota: si aparece error 429 RESOURCE_EXHAUSTED, reduce MAX_PLAYER_DOCUMENTS o espera a que se renueve la cuota de embeddings de Gemini


Total de documentos para indexar: 52
- Jugadores: 25
- Chunks glosario: 27


### 12.1 Prueba aislada de embeddings antes de indexar

Antes de enviar muchos documentos a ChromaDB, probamos una sola consulta de embeddings. Si esta celda falla, el problema es de modelo, API key o cuota; si funciona, podemos indexar por lotes.

In [50]:
# Probamos el modelo de embeddings con un unico texto corto.
test_embedding = embeddings.embed_query("prueba de embeddings para analitica de futbol")

# Si funciona, deberia devolver una lista de numeros.
print("Dimension del embedding:", len(test_embedding))
print("Primeros valores:", test_embedding[:5])

Dimension del embedding: 3072
Primeros valores: [-0.0043126647, 0.023887493, 0.024547875, -0.091807425, -0.00019322557]


### 12.2 Indexacion en ChromaDB

La primera ejecucion crea los embeddings y puede tardar. Despues, ChromaDB queda guardado en disco y se puede recargar desde `chroma_db`.

In [ ]:
import shutil
import hashlib
import time

# Nombre de la coleccion dentro de ChromaDB
COLLECTION_NAME = "asistente_analitica_futbol"

# Controla si quieres borrar la base vectorial y empezar desde cero
# Ponlo en True solo si quieres reconstruir ChromaDB completamente
RESET_CHROMA = False

# Si una ejecucion anterior fallo a medias, puedes dejar RESET_CHROMA=False para conservar lo ya indexado
if RESET_CHROMA and CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)
    # Nombre de la coleccion dentro de ChromaDB
    
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    # Modelo de embeddings de Gemini que convierte texto en vectores
    embedding_function=embeddings,
    # Carpeta donde se guardara la base vectorial
    persist_directory=str(CHROMA_PATH),
)
# Funcion auxiliar para crear IDs estables a partir del contenido y metadatos
def build_document_id(doc, index):
    # Combinamos fuente, tipo, jugador e indice para evitar IDs duplicados
    raw_id = "|".join([
        str(doc.metadata.get("source", "")),
        str(doc.metadata.get("doc_type", "")),
        str(doc.metadata.get("player", "")),
        str(index),
    ])
    # Convertimos ese texto en un hash corto y estable
    return hashlib.md5(raw_id.encode("utf-8")).hexdigest()

# Tamano de lote pequeno para no saturar la cuota de embeddings
BATCH_SIZE = 2

# Pausa entre lotes. Si vuelve a salir el error 429, hay que aumentar este valor
SLEEP_SECONDS = 20

# Creamos IDs para todos los documentos
document_ids = [build_document_id(doc, i) for i, doc in enumerate(rag_documents)]

# Consultamos IDs que ya existen para poder continuar si una ejecucion anterior se corto por cuota
existing = vectorstore._collection.get(include=[])
existing_ids = set(existing.get("ids", []))

# Insertamos los documentos por lotes
for start in range(0, len(rag_documents), BATCH_SIZE):
    # Calculamos el final del lote actual
    end = start + BATCH_SIZE

    # Seleccionamos documentos e IDs del lote
    batch_docs = rag_documents[start:end]
    batch_ids = document_ids[start:end]

    # Filtramos documentos que ya estaban indexados para no repetir embeddings ni gastar cuota
    pending_pairs = [
        (doc, doc_id)
        for doc, doc_id in zip(batch_docs, batch_ids)
        if doc_id not in existing_ids
    ]

    # Si todo el lote ya existe, lo saltamos
    if not pending_pairs:
        print(f"Lote {start + 1}-{min(end, len(rag_documents))} ya estaba indexado")
        continue

    # Separamos documentos e IDs pendientes
    pending_docs = [pair[0] for pair in pending_pairs]
    pending_ids = [pair[1] for pair in pending_pairs]

    try:
        # Añadimos el lote a ChromaDB. Aqui se calculan embeddings con Gemini
        vectorstore.add_documents(documents=pending_docs, ids=pending_ids)

        # Actualizamos la lista de IDs existentes
        existing_ids.update(pending_ids)

        # Mostramos progreso para saber por donde va la indexacion
        print(f"Indexados documentos {start + 1}-{min(end, len(rag_documents))} de {len(rag_documents)}")

    except Exception as error:
        # Si aparece cuota 429, paramos sin borrar lo ya indexado
        print("Se detuvo la indexacion por un error, probablemente cuota de Gemini.")
        print("Documentos guardados hasta ahora:", vectorstore._collection.count())
        print("Error:", error)
        break

    # Pausamos entre lotes para reducir riesgo de rate limit
    if end < len(rag_documents):
        time.sleep(SLEEP_SECONDS)

# Mostramos informacion de control.
print("Coleccion creada:", COLLECTION_NAME)
print("Directorio persistente:", CHROMA_PATH)
print("Documentos indexados:", vectorstore._collection.count())


Lote 1-2 ya estaba indexado
Lote 3-4 ya estaba indexado
Lote 5-6 ya estaba indexado
Lote 7-8 ya estaba indexado
Lote 9-10 ya estaba indexado
Lote 11-12 ya estaba indexado
Lote 13-14 ya estaba indexado
Lote 15-16 ya estaba indexado
Lote 17-18 ya estaba indexado
Lote 19-20 ya estaba indexado
Lote 21-22 ya estaba indexado
Lote 23-24 ya estaba indexado
Indexados documentos 25-26 de 52
Indexados documentos 27-28 de 52
Indexados documentos 29-30 de 52
Indexados documentos 31-32 de 52
Indexados documentos 33-34 de 52
Indexados documentos 35-36 de 52
Indexados documentos 37-38 de 52
Indexados documentos 39-40 de 52
Indexados documentos 41-42 de 52
Indexados documentos 43-44 de 52
Indexados documentos 45-46 de 52
Indexados documentos 47-48 de 52
Indexados documentos 49-50 de 52
Indexados documentos 51-52 de 52
Coleccion creada: asistente_analitica_futbol
Directorio persistente: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_L

### 12.3 Recarga de la coleccion existente

Cuando la base ya existe, puedes usar esta celda para recargarla sin volver a crear todos los embeddings.

In [66]:
# Esta celda permite recargar la coleccion ya creada sin recalcular embeddings.
vectorstore = Chroma(
    # Debe coincidir con el nombre usado al crear la coleccion.
    collection_name=COLLECTION_NAME,
    # Funcion de embeddings necesaria para transformar nuevas preguntas en vectores.
    embedding_function=embeddings,
    # Carpeta donde esta guardada la base vectorial.
    persist_directory=str(CHROMA_PATH),
)

# Confirmamos que la coleccion se ha cargado y cuantos documentos contiene.
print("Coleccion cargada:", COLLECTION_NAME)
print("Documentos disponibles:", vectorstore._collection.count())

Coleccion cargada: asistente_analitica_futbol
Documentos disponibles: 57


## 13. Prueba del retriever

Antes de crear el agente, comprobamos que la recuperacion funciona. Esta prueba es obligatoria en la practica: si el retriever falla, el agente tambien respondera mal.

In [67]:
# Convertimos ChromaDB en un retriever de LangChain.
retriever = vectorstore.as_retriever(
    # similarity significa que recupera los documentos mas parecidos semanticamente a la pregunta.
    search_type="similarity",
    # k=5 indica que devolvera los 5 documentos mas relevantes.
    search_kwargs={"k": 5},
)

# Funcion auxiliar para probar que el retriever encuentra documentos relevantes.
def probar_retriever(pregunta):
    # Enviamos la pregunta al retriever y recuperamos documentos relacionados.
    docs = retriever.invoke(pregunta)

    # Mostramos la pregunta original.
    print("Pregunta:", pregunta)

    # Mostramos cuantos documentos se recuperaron.
    print("Documentos recuperados:", len(docs))

    # Recorremos los documentos recuperados para inspeccionarlos.
    for i, doc in enumerate(docs, start=1):
        print("" + "=" * 80)
        print(f"Documento {i}")

        # Mostramos metadatos utiles para saber de donde viene la informacion.
        print("Fuente:", doc.metadata.get("source"))
        print("Tipo:", doc.metadata.get("doc_type"))

        # Si el documento es de un jugador, mostramos el nombre.
        if doc.metadata.get("player"):
            print("Jugador:", doc.metadata.get("player"))

        # Imprimimos solo el inicio del contenido para no saturar la salida.
        print(doc.page_content[:900])

    # Devolvemos los documentos por si queremos usarlos en otra celda.
    return docs

In [68]:
# Prueba conceptual: deberia recuperar chunks del glosario donde se explica xG.
docs_prueba_metricas = probar_retriever("Que significa xG y como se interpreta en scouting?")

Pregunta: Que significa xG y como se interpreta en scouting?
Documentos recuperados: 5
Documento 1
Fuente: glosario_metricas_futbol.md
Tipo: metric_glossary
## Goles esperados y creatividad esperada

### xG

Expected Goals o goles esperados.

Representa la probabilidad acumulada de que los tiros de un jugador terminen en gol segun la calidad de las ocasiones.

Interpretacion:

- xG alto: el jugador recibe o genera ocasiones claras de remate.
- Goles mayores que xG: el jugador ha finalizado por encima de lo esperado.
- Goles menores que xG: el jugador ha finalizado por debajo de lo esperado.

Uso analitico: clave para evaluar delanteros, extremos y mediapuntas.

### npxG

Non-penalty expected goals o goles esperados sin penaltis.

Uso analitico: mejor que xG total para comparar capacidad de generar ocasiones en juego abierto.

### xAG

Expected Assisted Goals.

Mide la calidad de las ocasiones que un jugador crea para sus companeros mediante pases que terminan en tiro.

Uso analitico: c

In [69]:
# Prueba aplicada: deberia recuperar la ficha de Mohamed Salah y posiblemente metricas relacionadas.
docs_prueba_jugador = probar_retriever("Compara el perfil ofensivo de Mohamed Salah usando xG xAG progresiones y creacion")

Pregunta: Compara el perfil ofensivo de Mohamed Salah usando xG xAG progresiones y creacion
Documentos recuperados: 5
Documento 1
Fuente: players_data-2024_2025.csv
Tipo: player_scouting_profile
Jugador: Mohamed Salah
Ficha de scouting de Mohamed Salah

Contexto:
- Jugador: Mohamed Salah
- Equipo: Liverpool
- Liga: eng Premier League
- Posicion: FW

Metricas disponibles:
- Player: Mohamed Salah
- Nation: eg EGY
- Pos: FW
- Squad: Liverpool
- Comp: eng Premier League
- Age: 32.0
- Born: 1992.0
- MP: 38
- Starts: 38
- Min: 3371
- 90s: 37.5
- Gls: 29
- Ast: 18
- G+A: 47
- G-PK: 20
- PK: 9
- PKatt: 9
- xG: 25.2
- npxG: 18.2
- xAG: 14.2
- xA: 9.1
- npxG+xAG: 32.4
- xG+xAG: 1.05
- Sh: 121
- SoT: 50
- SoT%: 41.3
- Sh/90: 3.23
- SoT/90: 1.33
- G/Sh: 0.17
- G/SoT: 0.4
- Dist: 14.5
- npxG/Sh: 0.15
- G-xG: 3.8
- np:G-xG: 1.8
- Cmp: 892
- Att: 1263
- Cmp%: 70.6
- TotDist: 13343
- PrgDist: 3919
- KP: 88
- 1/3: 53
- PPA: 93
- CrsPA: 15
- PrgP: 144
- SCA: 169
- SCA90: 4.51
- GCA: 27
- GCA90: 0.72
- T

## 14. RAG basico con Gemini

Ahora conectamos el retriever con Gemini. Esta version todavia no usa LangGraph ni memoria; sirve para comprobar que el modelo responde usando contexto recuperado desde ChromaDB.

In [70]:
# Funcion para convertir documentos recuperados en texto legible para el prompt.
def format_retrieved_docs(docs):
    # Creamos una lista donde guardaremos cada documento formateado.
    formatted = []

    # Recorremos los documentos recuperados por ChromaDB.
    for i, doc in enumerate(docs, start=1):
        # Extraemos metadatos utiles.
        source = doc.metadata.get("source", "fuente desconocida")
        doc_type = doc.metadata.get("doc_type", "tipo desconocido")
        player = doc.metadata.get("player")

        # Creamos una cabecera para saber de donde viene cada fragmento.
        header = f"[Documento {i} | fuente: {source} | tipo: {doc_type}"
        if player:
            header += f" | jugador: {player}"
        header += "]"

        # Juntamos cabecera y contenido del documento.
        formatted.append(header + "" + doc.page_content)

    # Devolvemos todos los documentos separados por una linea visual.
    return "---".join(formatted)


# Funcion principal de RAG basico.
def responder_con_rag_basico(pregunta, k=5):
    # Recuperamos documentos relevantes desde ChromaDB.
    docs = retriever.invoke(pregunta)

    # Nos quedamos con los k primeros por si el retriever devuelve mas.
    docs = docs[:k]

    # Convertimos los documentos recuperados en texto para Gemini.
    contexto = format_retrieved_docs(docs)

    # Creamos el prompt final con instrucciones, pregunta y contexto.
    prompt = f"""
{SYSTEM_PROMPT}

Pregunta del usuario:
{pregunta}

Contexto recuperado desde ChromaDB:
{contexto}

Instrucciones para responder:
- Responde usando el contexto recuperado.
- Si el contexto no contiene la informacion necesaria, dilo claramente.
- Cita brevemente las fuentes usadas indicando si vienen del glosario o de una ficha de jugador.
""".strip()

    # Enviamos el prompt a Gemini.
    respuesta = llm.invoke(prompt)

    # Devolvemos respuesta y documentos para poder inspeccionar el RAG.
    return {
        "respuesta": respuesta.content,
        "documentos": docs,
        "prompt": prompt,
    }

### 14.1 Prueba conceptual del RAG

Esta prueba debe recuperar fragmentos del glosario y responder usando definiciones de metricas.

In [73]:
resultado_xg = responder_con_rag_basico("Que significa xG y como se interpreta en scouting?")
print(resultado_xg["respuesta"])

El `xG` (Expected Goals o goles esperados) representa la probabilidad acumulada de que los tiros de un jugador terminen en gol, basándose en la calidad de las ocasiones generadas (fuente: glosario_metricas_futbol.md).

En el scouting, se interpreta de la siguiente manera:

*   **xG alto**: Indica que el jugador recibe o genera ocasiones claras de remate, lo que sugiere una buena capacidad para posicionarse o encontrar oportunidades de tiro.
*   **Goles mayores que xG**: Significa que el jugador ha finalizado por encima de lo esperado, lo que podría indicar una excelente capacidad de definición o un período de alta eficacia.
*   **Goles menores que xG**: Sugiere que el jugador ha finalizado por debajo de lo esperado, lo que podría indicar una mala racha de finalización o que necesita mejorar su puntería.

Analíticamente, el `xG` es una métrica clave para evaluar a delanteros, extremos y mediapuntas, ya que ayuda a entender su contribución ofensiva más allá de los goles reales (fuente: g

### 14.2 Inspeccion de documentos usados

Esta celda permite demostrar que la respuesta no sale solo del LLM, sino de documentos recuperados por ChromaDB.

In [74]:
for i, doc in enumerate(resultado_xg["documentos"], start=1):
    print("=" * 80)
    print(f"Documento recuperado {i}")
    print("Metadatos:", doc.metadata)
    print(doc.page_content[:800])

Documento recuperado 1
Metadatos: {'source': 'glosario_metricas_futbol.md', 'doc_type': 'metric_glossary'}
## Goles esperados y creatividad esperada

### xG

Expected Goals o goles esperados.

Representa la probabilidad acumulada de que los tiros de un jugador terminen en gol segun la calidad de las ocasiones.

Interpretacion:

- xG alto: el jugador recibe o genera ocasiones claras de remate.
- Goles mayores que xG: el jugador ha finalizado por encima de lo esperado.
- Goles menores que xG: el jugador ha finalizado por debajo de lo esperado.

Uso analitico: clave para evaluar delanteros, extremos y mediapuntas.

### npxG

Non-penalty expected goals o goles esperados sin penaltis.

Uso analitico: mejor que xG total para comparar capacidad de generar ocasiones en juego abierto.

### xAG

Expected Assisted Goals.

Mide la calidad de las ocasiones que un jugador crea para sus companeros mediante pa
Documento recuperado 2
Metadatos: {'doc_type': 'metric_glossary', 'source': 'glosario_metric

### 14.3 Prueba sobre jugador

Esta prueba comprueba si el RAG recupera una ficha de jugador cuando la pregunta menciona un futbolista indexado.

In [75]:
# Cambia el nombre por uno que aparezca en tus player_documents si quieres probar otro jugador.
resultado_jugador = responder_con_rag_basico("Analiza brevemente el perfil de Mile Svilar usando sus metricas de portero")
print(resultado_jugador["respuesta"])

Analizando el perfil de Mile Svilar (Documento 1 | tipo: player_scouting_profile), podemos destacar los siguientes aspectos de su rendimiento como portero en la Serie A italiana con la Roma, durante 38 partidos y 3420 minutos jugados:

*   **Goles Encajados (GA) y Goles Encajados por 90 minutos (GA90):** Svilar ha encajado 35 goles en total, lo que se traduce en un promedio de 0.92 goles por cada 90 minutos. Este es un buen registro, indicando una solidez defensiva.
*   **Porcentaje de Paradas (Save%):** Con 117 paradas de 151 tiros a puerta recibidos (SoTA), su porcentaje de paradas es del 79.5%. Este es un valor muy alto y sugiere una gran capacidad para detener disparos.
*   **Porterías a Cero (CS) y Porcentaje de Porterías a Cero (CS%):** Ha mantenido su portería a cero en 16 partidos, lo que representa un 42.1% de los encuentros. El **CS%** (Documento 3 | tipo: metric_glossary) es un indicador de la solidez defensiva del equipo, pero también de la contribución del portero. Un 42.1

## Resultado de esta fase

En este punto ya tenemos:

- Filas del CSV convertidas en documentos textuales de scouting.
- Glosario dividido en chunks semanticos.
- Embeddings creados con Gemini.
- Base vectorial persistente en ChromaDB.
- Retriever probado antes de conectar el agente.

El siguiente paso sera construir una cadena RAG basica y despues envolverla en un agente LangGraph con memoria conversacional.